# 첫 신경망 — 손글씨 숫자 분류기 (MLP)

> ⏱ 50분 · CPU로 충분

**목표:** 진짜 데이터셋으로 신경망을 처음부터 끝까지 학습시킵니다. 여기서 만드는 **학습 루프**는 LLM 학습에서도 거의 그대로 쓰입니다.

## 데이터: MNIST

28×28 흑백 손글씨 숫자 이미지 7만 장. `Dataset`은 "i번째 샘플을 돌려주는 것", `DataLoader`는 "그걸 배치로 묶고 섞어주는 것"입니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tf = transforms.ToTensor()   # 이미지를 0~1 사이 텐서로
train_ds = datasets.MNIST("data", train=True, download=True, transform=tf)
test_ds = datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)

x, y = next(iter(train_dl))
print(x.shape, y.shape)   # [64, 1, 28, 28] 이미지 64장, [64] 정답 라벨 64개

**코드 읽기**

- `device = "cuda" if ... else "cpu"` — GPU가 있으면 쓰고 없으면 CPU. 이 한 줄 덕분에 같은 코드가 Colab GPU에서도, 내 노트북에서도 돌아갑니다. 이후 모델과 데이터에 `.to(device)`를 붙이는 것이 규칙입니다.
- `torchvision` — 이미지용 데이터셋·모델·변환을 모아 둔 PyTorch의 자매 라이브러리. MNIST를 직접 내려받아 파싱하는 코드를 안 짜도 됩니다.
- `transforms.ToTensor()` — 이미지 파일(0~255 정수 픽셀)을 0~1 실수 텐서 `[채널, 높이, 너비]`로 바꿉니다. 왜 0~1로 줄이나: 입력값이 크면 가중치 곱의 결과가 커져 학습이 불안정해집니다. 1부의 `StandardScaler`와 같은 목적입니다.
- `datasets.MNIST("data", train=True, download=True, transform=tf)` — `train=True`면 학습용 6만 장, `False`면 평가용 1만 장. 데이터셋 제작자가 이미 나눠 두었으므로 `train_test_split`이 필요 없습니다. `transform`은 "샘플을 꺼낼 때마다 이 함수를 적용하라"입니다.
- `Dataset` 객체 — `len(ds)`와 `ds[i]`("i번째 샘플과 정답을 달라")만 제공합니다. 데이터가 디스크에 있든 메모리에 있든 이 인터페이스만 맞추면 되므로, 내 데이터를 쓸 때도 이 두 메서드만 구현하면 됩니다.
- `DataLoader(ds, batch_size=64, shuffle=True)` — 샘플을 64개씩 묶어 텐서 하나로 만들어 줍니다. 왜 배치인가: 6만 장을 한 번에 넣으면 메모리가 모자라고, 1장씩 넣으면 GPU가 놀고 기울기가 너무 요동칩니다. 64개 정도가 둘의 절충입니다. `shuffle=True`는 에폭마다 순서를 섞어 모델이 "순서"를 외우거나 비슷한 샘플이 몰려 오는 것을 막습니다. 평가용 로더에는 섞을 이유가 없어 빼고, 기울기 계산이 없어 메모리 여유가 있으므로 배치를 256으로 키웠습니다.
- `next(iter(train_dl))` — 배치 하나만 꺼내 봅니다. 학습 전에 **데이터 모양을 확인**하는 습관입니다. `[64, 1, 28, 28]`은 배치 64, 채널 1(흑백), 28×28 픽셀입니다.
## 모델: `nn.Module`로 직접 정의하기

모델은 `__init__`에서 **층을 선언**하고, `forward`에서 **데이터가 흐르는 순서**를 적습니다. 모델을 수정한다는 건 결국 이 두 곳을 고치는 일입니다.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),              # [B,1,28,28] → [B,784]
            nn.Linear(784, hidden),
            nn.ReLU(),                 # 비선형 함수. 이게 없으면 층을 쌓아도 직선 하나와 같음
            nn.Linear(hidden, 10),     # 숫자 0~9 각각의 점수(logit)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))

**코드 읽기**

- `class MLP(nn.Module)` — PyTorch의 모든 모델은 `nn.Module`을 상속합니다. 상속하면 ① 안에 선언한 층들의 파라미터를 `model.parameters()`로 한꺼번에 모을 수 있고, ② `model(x)`라고 부르면 `forward`가 실행되고, ③ `.to(device)`, `.eval()`, `state_dict()` 같은 기능이 공짜로 따라옵니다. 이 규칙만 지키면 어떤 구조든 만들 수 있습니다.
- `super().__init__()` — 부모 클래스(`nn.Module`)의 초기화를 먼저 실행. 빼먹으면 층을 등록하는 장치가 준비되지 않아 에러가 납니다. 반드시 첫 줄에 씁니다.
- `nn.Sequential(...)` — 층들을 순서대로 이어 붙이는 상자. 입력이 위에서 아래로 차례로 통과합니다. 갈래가 없는 단순한 구조는 이걸로 충분하고, `forward`도 한 줄이 됩니다.
- `nn.Flatten()` — `[64, 1, 28, 28]`을 `[64, 784]`로 폅니다. 왜: `nn.Linear`는 벡터를 받기 때문입니다(1부에서 `digits.data`가 64개짜리 벡터였던 것과 같음). 배치 차원(64)은 그대로 두고 나머지만 곱해서 폅니다.
- `nn.Linear(784, hidden)` — 784개 입력 → `hidden`개 출력인 가중합. 안에 `784 × hidden` 크기의 가중치 행렬과 `hidden`개의 편향이 있습니다. 1부의 로지스틱 회귀가 정확히 `nn.Linear(784, 10)` 하나입니다.
- `nn.ReLU()` — `max(0, x)`. 왜 하필 ReLU인가: 계산이 거의 공짜이고, 기울기가 0 아니면 1이라 깊게 쌓아도 기울기가 사라지지 않습니다. 옛날의 sigmoid/tanh는 양 끝에서 기울기가 0에 가까워져 깊은 모델 학습이 어려웠습니다. 특별한 이유가 없으면 ReLU(트랜스포머에서는 그 변형인 GELU)를 씁니다.
- `nn.Linear(hidden, 10)` — 마지막 층은 클래스 수(10)만큼 출력합니다. 이 출력이 **로짓(logit)**, 즉 정규화되지 않은 점수입니다. 확률로 바꾸는 softmax는 여기 없고 손실 함수 안에 들어 있습니다(아래 참고).
- 왜 `hidden=128`인가: 정해진 답은 없습니다. 784와 10 사이의 적당한 크기이고, 실습 1번에서 직접 바꿔 보게 됩니다.
- `sum(p.numel() for p in model.parameters())` — `numel`은 텐서 안의 원소 수. 파라미터 수를 세는 관용구입니다. 784×128 + 128 + 128×10 + 10 = 101,770. 앞으로 모델을 볼 때마다 이 숫자를 먼저 확인하세요. "0.5B 모델"이라는 말이 이 숫자입니다.

## 학습 루프와 평가

In [ ]:
def evaluate(model, dl):
    model.eval()                         # 평가 모드 (dropout 등이 꺼짐)
    correct = 0
    with torch.no_grad():                # 평가 땐 기울기 계산 불필요 → 빠르고 메모리 절약
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1) == y).sum().item()
    return correct / len(dl.dataset)

def train(model, epochs=3, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()      # 분류 문제의 표준 손실
    for epoch in range(epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)   # 데이터도 모델과 같은 장치로
            loss = loss_fn(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch+1}  마지막 배치 loss {loss.item():.4f}  테스트 정확도 {evaluate(model, test_dl):.4f}")

print("학습 전 정확도:", evaluate(model, test_dl))   # 약 0.1 (찍는 수준)
train(model)

3 에폭이면 97% 안팎이 나옵니다. 방금 신경망을 하나 학습시켰습니다.

**코드 읽기**

- `model.eval()` / `model.train()` — 모드 전환. 지금 모델에는 영향이 없지만, 뒤에 나올 dropout이나 batch norm은 학습 때와 평가 때 동작이 다릅니다. 평가 함수 첫 줄에 `eval()`, 학습 루프 첫 줄에 `train()`을 쓰는 것을 지금부터 습관으로 만드세요. 빠뜨리면 조용히 성능이 나빠지는 흔한 버그입니다.
- `with torch.no_grad():` — 평가 때는 파라미터를 갱신하지 않으므로 계산 그래프를 기록할 필요가 없습니다. 끄면 메모리를 절반 이하로 쓰고 속도도 빨라집니다.
- `model(x).argmax(dim=1)` — 출력 `[256, 10]`에서 각 행(샘플)마다 점수가 가장 큰 열(클래스)의 번호를 고릅니다. `dim=1`이 "열 방향으로 최대"입니다. 이것이 예측한 숫자입니다.
- `(pred == y).sum().item()` — 맞힌 개수. `==`는 원소별 비교로 True/False 텐서를 만들고, `sum()`이 True의 개수를 셉니다.
- `torch.optim.Adam(model.parameters(), lr=1e-3)` — 앞 레슨의 SGD 대신 Adam. 왜: SGD는 모든 파라미터에 같은 학습률을 쓰지만, Adam은 파라미터마다 최근 기울기의 크기를 보고 걸음 크기를 자동 조절합니다. 학습률에 덜 민감해서 처음 시도할 때 거의 실패하지 않습니다. `1e-3`은 Adam의 관례적 기본값입니다.
- `nn.CrossEntropyLoss()` — 분류의 표준 손실. 로짓을 받아 안에서 softmax로 확률을 만들고, **정답 클래스에 부여한 확률의 −log**를 냅니다. 정답 확률이 1이면 0, 0.1이면 2.3, 0.01이면 4.6으로, 확신을 갖고 틀릴수록 벌점이 급격히 커집니다. 회귀의 MSE를 여기 쓰면 안 되는 이유: 클래스 번호 3과 7의 "거리"에는 아무 의미가 없기 때문입니다.
- 왜 softmax를 모델에 넣지 않고 손실 안에 두나: 수치 안정성 때문입니다. softmax 뒤에 log를 따로 계산하면 아주 작은 확률에서 −무한대가 나올 수 있는데, 둘을 합쳐 계산하면 안전합니다. PyTorch의 관례이므로 **모델 출력은 로짓**으로 두세요.
- `x.to(device), y.to(device)` — 배치를 꺼낼 때마다 GPU로 옮깁니다. 데이터셋 전체를 GPU에 올리지 않는 이유는 메모리 때문입니다. 모델과 데이터가 다른 장치에 있으면 "expected all tensors on same device" 에러가 납니다. 이 에러를 보면 `.to(device)` 누락을 찾으세요.
- 루프 구조 — 바깥 `for epoch`는 데이터 전체를 몇 바퀴 돌지, 안쪽 `for x, y in train_dl`은 배치 단위 스텝입니다. 안쪽의 네 줄(`loss`, `zero_grad`, `backward`, `step`)이 앞 레슨과 완전히 같다는 것을 확인하세요.
- 에폭마다 `evaluate`를 부르는 이유 — 학습이 진행되며 성능이 어떻게 변하는지 봐야 "얼마나 돌려야 하는지", "과적합이 시작됐는지"를 알 수 있습니다. [학습 잘 시키는 법](https://yun-sooyong.github.io/dl-study/#23-training-recipes)에서 이것을 그래프로 발전시킵니다.

## 모델이 틀린 것 들여다보기

In [ ]:
import matplotlib.pyplot as plt

x, y = next(iter(test_dl))
pred = model(x.to(device)).argmax(dim=1).cpu()
wrong = (pred != y).nonzero().flatten()[:8]
fig, axes = plt.subplots(1, max(len(wrong), 1), figsize=(2 * max(len(wrong), 1), 2.5), squeeze=False)
for ax, i in zip(axes[0], wrong):
    ax.imshow(x[i, 0], cmap="gray"); ax.axis("off")
    ax.set_title(f"pred {pred[i].item()} / true {y[i].item()}")
plt.show()

**코드 읽기**

- `.cpu()` — 그림을 그리거나 numpy로 바꾸려면 텐서가 CPU에 있어야 합니다. GPU 텐서를 바로 `imshow`에 넣으면 에러가 납니다.
- `(pred != y).nonzero().flatten()` — 틀린 위치의 인덱스만 뽑습니다. `nonzero`는 True인 위치를 `[개수, 1]` 모양으로 주기 때문에 `flatten`으로 1차원으로 폅니다. `[:8]`로 앞의 8개만.
- `plt.subplots(..., squeeze=False)` — 틀린 것이 1개뿐이면 `axes`가 배열이 아니라 단일 객체가 되어 `zip`에서 에러가 납니다. `squeeze=False`는 항상 2차원 배열로 받겠다는 뜻이고, 그래서 `axes[0]`로 첫 행을 꺼냅니다. 이런 사소한 경계 조건이 실제 코드에서 자주 발목을 잡습니다.
- 왜 틀린 것을 굳이 보나: 97%라는 숫자는 "어떤 3%인지"를 말해 주지 않습니다. 사람이 봐도 헷갈리는 글씨라면 모델 탓이 아니고, 명백한 숫자를 틀린다면 모델이나 데이터에 문제가 있다는 뜻입니다. 지표 뒤의 실제 샘플을 보는 습관은 5부까지 계속 강조됩니다.

## 핵심 정리

- `Dataset`은 샘플 하나를, `DataLoader`는 섞인 배치를 돌려줍니다.
- 모델은 `nn.Module`: `__init__`에 층을 선언하고 `forward`에 흐름을 적습니다.
- 분류 문제의 손실은 `CrossEntropyLoss`, 출력은 클래스별 점수(logit)입니다.
- 평가할 때는 `model.eval()` + `torch.no_grad()`, 그리고 **학습에 쓰지 않은 데이터**로 합니다.
- 모델과 데이터는 같은 device에 있어야 합니다.

## 스스로 점검

답을 머릿속으로 먼저 말해 본 뒤 펼쳐 보세요.

<details><summary>Q1. ReLU 같은 활성화 함수를 빼면 왜 층을 쌓는 의미가 없어지나요?</summary>

Linear 두 개를 연달아 적용하면 `W2(W1x + b1) + b2`로, 결국 또 하나의 Linear와 같습니다. 비선형 함수가 사이에 있어야 층을 쌓을수록 더 복잡한 함수를 표현할 수 있습니다.

</details>

<details><summary>Q2. 학습 전 정확도가 약 10%인 이유는?</summary>

가중치가 무작위라 10개 클래스 중 하나를 찍는 것과 같기 때문입니다. 학습 전 수치를 확인해 두면 '정말 학습이 되고 있는지' 판단하는 기준이 됩니다.

</details>

<details><summary>Q3. 학습 데이터의 정확도가 아니라 <b>테스트</b> 데이터의 정확도를 보는 이유는?</summary>

우리가 원하는 것은 처음 보는 데이터에서도 잘하는 것(일반화)입니다. 학습 데이터 정확도는 외우기만 해도 올라갑니다.

</details>

## 직접 고쳐보기

1. `MLP(hidden=16)`과 `MLP(hidden=512)`를 각각 학습시켜 정확도와 파라미터 수를 비교하세요.
2. `nn.ReLU()`를 지우고 학습시켜 보세요. 정확도가 어떻게 되나요? 왜일까요?
3. 은닉층을 하나 더 추가해 보세요 (`Linear → ReLU → Linear → ReLU → Linear`). shape이 맞도록 숫자를 직접 맞춰야 합니다.
4. `lr=1e-1`, `lr=1e-5`로 바꿔보세요.
5. (도전) 학습 루프에서 100 스텝마다 loss를 리스트에 모아 `plt.plot`으로 그려보세요. 앞으로 계속 보게 될 **loss 곡선**입니다.